# 🌾 Paddy-YOLO: Rice Pest Detection on Kaggle GPU

This notebook trains and evaluates **Paddy-YOLO** (Computers and Electronics in Agriculture, 2025) on Kaggle using the **R2000** dataset (`dmrashidferrer/r2000v1-2024-images`).

### Key Features:
- **Backbone**: YOLOv8-N with C2f modules replaced by **RGG** (RepConv + dual GhostConv with 5x5 DWConv).
- **Loss**: Dynamic Inner-MPDIoU (**DIMIoU**, Strategy H cosine dynamic ratio scheduling).
- **Cross-Validation**: 5-Fold Cross Validation (3 train, 1 val, 1 test folds).

In [ ]:
# 1. Verify GPU Accelerator
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Please select: Notebook settings -> Accelerator -> GPU (T4 x2 or P100)')

In [ ]:
# 2. Install dependencies & run Paddy-YOLO sanity check
!pip install ultralytics -q
!python verify_paddy_yolo.py --smoke-test

In [ ]:
# 3. Discover Kaggle R2000 Dataset & Generate 5-Fold Splits
import os
from pathlib import Path
from paddy_yolo.dataset.prepare_r2000 import prepare_kaggle_r2000, create_5fold_splits

# Auto-detect mounted dataset in /kaggle/input
input_candidates = list(Path('/kaggle/input').glob('*r2000*'))
if input_candidates:
    raw_dir = input_candidates[0]
    print('Found R2000 dataset at:', raw_dir)
else:
    raw_dir = Path('/kaggle/input/r2000v1-2024-images')
    print('Using default path:', raw_dir)

output_dir = Path('/kaggle/working/data/r2000')
if raw_dir.exists():
    prepare_kaggle_r2000(input_dir=raw_dir, output_dir=output_dir)
    folds = create_5fold_splits(dataset_dir=output_dir)
    print('Generated 5-Fold Cross Validation Configs:')
    for f in folds:
        print('  ->', f)
else:
    print('⚠️ Dataset not found. Please click "Add Input" -> search "r2000v1-2024-images".')

In [ ]:
# 4. Train Paddy-YOLO on Fold 0 with Paper Hyperparameters
# Table 6: 300 epochs, AdamW, lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005
!python train.py \
    --model configs/paddy-yolo.yaml \
    --data /kaggle/working/data/r2000/data_fold0.yaml \
    --hyp configs/hyp_paper.yaml \
    --epochs 300 \
    --batch 32 \
    --imgsz 640 \
    --device 0 \
    --strategy H \
    --project /kaggle/working/runs/train \
    --name paddy_yolo_fold0

In [ ]:
# 5. Evaluate Best Checkpoint on Test Set (mAP@50, mAP@50:95, F1-Score)
weights = '/kaggle/working/runs/train/paddy_yolo_fold0/weights/best.pt'
!python val.py \
    --weights {weights} \
    --data /kaggle/working/data/r2000/data_fold0.yaml \
    --split test \
    --device 0

In [ ]:
# 6. Predict and Visualize Pest Detections
import glob
from PIL import Image
import matplotlib.pyplot as plt

test_images = glob.glob('/kaggle/working/data/r2000/images/*.jpg')
if test_images:
    sample_img = test_images[0]
    !python predict.py \
        --weights {weights} \
        --source {sample_img} \
        --device 0 \
        --output /kaggle/working/runs/predict
    
    preds = glob.glob('/kaggle/working/runs/predict/*.jpg')
    if preds:
        plt.figure(figsize=(10, 10))
        plt.imshow(Image.open(preds[0]))
        plt.axis('off')
        plt.title('Paddy-YOLO Pest Detection Result')
        plt.show()